# 🔍 Data Quality Assessment (DQA) - Robotaxi Finance Analytics

Notebook ini dibuat untuk mendeteksi, mendokumentasikan, dan menganalisis masalah kualitas data (*data quality issues*) di seluruh 8 dataset proyek **Finance of Robotaxi**. 

### Fokus Pemeriksaan:
1. **Duplikasi Data:** Memeriksa baris duplikat, terutama pada kolom ID unik.
2. **Integritas Referensial (Keys Match):** Memeriksa apakah ID kunci asing (*Foreign Keys*) cocok dengan tabel referensi induknya.
3. **Kejanggalan & Anomali Logika:** Memeriksa logika bisnis seperti:
   - Apakah ada tanggal mulai yang setelah tanggal selesai?
   - Apakah ada nilai biaya negatif?
   - Memeriksa keselarasan data kendaraan di `ds1_vehicles` vs `ds3_fleet_vehicles`.

In [1]:
import pandas as pd
import numpy as np
import os

# Definisikan path ke data
data_dir = r".."  # Karena notebook berada di dalam folder 'notebooks'
if not os.path.exists(os.path.join(data_dir, 'ds1_trips.csv')):
    data_dir = r"c:\Users\hpvic\OneDrive\Documents\Finance of Robotaxi"

print(f"Direktori data: {data_dir}")

Direktori data: ..


## 1. Memuat Semua Dataset

In [2]:
trips = pd.read_csv(os.path.join(data_dir, 'ds1_trips.csv'))
vehicles_ds1 = pd.read_csv(os.path.join(data_dir, 'ds1_vehicles.csv'))

customers = pd.read_csv(os.path.join(data_dir, 'ds2_customers.csv'))
transactions = pd.read_csv(os.path.join(data_dir, 'ds2_transactions.csv'))

fleet_vehicles_ds3 = pd.read_csv(os.path.join(data_dir, 'ds3_fleet_vehicles.csv'))
maintenance = pd.read_csv(os.path.join(data_dir, 'ds3_maintenance_records.csv'))

incidents = pd.read_csv(os.path.join(data_dir, 'ds4_incidents.csv'))
insurance = pd.read_csv(os.path.join(data_dir, 'ds4_insurance_policies.csv'))

print("Semua dataset berhasil dimuat!")

Semua dataset berhasil dimuat!


## 2. Pemeriksaan Duplikasi ID
Mari kita verifikasi mengapa jumlah ID unik lebih kecil dari total baris (5.000) pada beberapa tabel.

In [3]:
def check_duplicates(df, id_col, name):
    total_rows = len(df)
    unique_ids = df[id_col].nunique()
    duplicate_count = total_rows - unique_ids
    print(f"=== {name} ({id_col}) ===")
    print(f"Total Baris: {total_rows}")
    print(f"ID Unik: {unique_ids}")
    print(f"Jumlah Duplikat ID: {duplicate_count}")
    if duplicate_count > 0:
        # Tampilkan contoh data duplikat
        dup_ids = df[df.duplicated(subset=[id_col], keep=False)][id_col].head(4).unique()
        print("Contoh data dengan ID duplikat:")
        display(df[df[id_col].isin(dup_ids)].sort_values(by=id_col).head(4))
    print("\n")

check_duplicates(trips, 'trip_id', 'Trips (ds1)')
check_duplicates(transactions, 'transaction_id', 'Transactions (ds2)')
check_duplicates(maintenance, 'record_id', 'Maintenance Records (ds3)')
check_duplicates(incidents, 'incident_id', 'Incidents (ds4)')

=== Trips (ds1) (trip_id) ===
Total Baris: 5000
ID Unik: 4979
Jumlah Duplikat ID: 21
Contoh data dengan ID duplikat:


,trip_id,vehicle_id,customer_id,trip_start_time,trip_end_time,pickup_lat,pickup_lon,dropoff_lat,dropoff_lon,distance_km,fare_amount_usd,surge_multiplier,payment_method,trip_status,passenger_rating,cancellation_reason,route_type,weather_condition,traffic_level,discount_applied_usd
326,162674,417925,384592,2024-05-18 07:22:52,2024-02-13 17:50:49,44.942272,-86.085938,31.393148,-77.000248,15.98,85.62,1.0,Google Pay,Completed,5,NaN,Shortest,Cloudy,Low,0
4896,162674,314619,782118,2024-08-24 16:13:42,2025-01-28 00:39:37,29.633122,-112.531537,30.325889,-77.954963,2.58,12.09,2.0,Debit Card,Completed,5,Passenger Cancel,Highway,Clear,High,0
4185,192651,557516,216929,2023-09-30 12:32:59,2024-07-18 18:43:22,25.112054,-96.690223,44.681213,-121.753129,5.59,63.25,1.0,Credit Card,Completed,5,NaN,Avoid Tolls,Clear,Severe,10
62,192651,694538,662383,2024-01-19 08:47:15,2023-04-27 01:41:17,45.516779,-108.663514,26.711576,-110.315323,0.59,17.70,2.0,PayPal,In Progress,5,NaN,Scenic,Clear,Medium,0




=== Transactions (ds2) (transaction_id) ===
Total Baris: 5000
ID Unik: 4986
Jumlah Duplikat ID: 14
Contoh data dengan ID duplikat:


,transaction_id,customer_id,zone_id,transaction_date,gross_amount_usd,tax_amount_usd,tip_amount_usd,refund_amount_usd,payment_method,payment_status,promo_code_used,discount_pct,platform_fee_usd,driver_payout_usd,net_revenue_usd,currency,billing_cycle,invoice_number,reconciliation_status,dispute_flag
892,301030,456762,719432,2024-03-07,103.07,14.26,0,0,Credit Card,Settled,NEWUSER,15,2.52,10.81,29.08,USD,Per Trip,1645023,Unreconciled,0
1660,301030,645578,559914,2025-04-07,53.74,12.17,0,0,PayPal,Refunded,NONE,10,2.30,57.98,27.39,USD,Per Trip,4945237,Reconciled,0
3552,482720,306673,609492,2023-06-28,63.55,7.67,0,0,Debit Card,Settled,SAVE20,0,2.37,66.22,36.03,USD,Per Trip,3668071,Reconciled,0
1033,482720,813032,353987,2024-01-22,13.71,2.99,0,0,Apple Pay,Pending,SAVE20,20,1.34,69.43,13.52,USD,Per Trip,1278999,Reconciled,0




=== Maintenance Records (ds3) (record_id) ===
Total Baris: 5000
ID Unik: 4988
Jumlah Duplikat ID: 12
Contoh data dengan ID duplikat:


,record_id,fleet_vehicle_id,technician_id,service_date,service_type,parts_cost_usd,labor_cost_usd,total_cost_usd,downtime_hours,mileage_at_service,next_service_mileage,warranty_claim,fault_code,severity_level,repair_status,vendor_name,invoice_number,approval_status,estimated_hours,actual_hours
374,147678,617914,759820,2025-02-14,Tire Rotation,191.92,142.94,293.42,0.9,171094,110301,0,F010,Medium,Completed,TechWrench,2118552,Approved,6.1,12.0
1769,147678,110077,186665,2022-11-18,Routine Check,118.22,20.23,624.02,1.0,22411,55155,0,F001,Low,Completed,EV Specialists,5241981,Pending,4.2,18.4
1068,590192,109809,110219,2023-07-06,Sensor Calibration,358.20,70.62,311.19,7.1,69907,312881,0,NONE,Medium,Completed,FleetCare Inc,5656424,Approved,7.9,17.0
671,590192,178865,578635,2024-03-05,Software Update,11.81,73.51,494.71,0.9,188407,212961,0,F002,High,Cancelled,FleetCare Inc,1125559,Rejected,15.0,4.9




=== Incidents (ds4) (incident_id) ===
Total Baris: 5000
ID Unik: 4987
Jumlah Duplikat ID: 13
Contoh data dengan ID duplikat:


,incident_id,policy_id,geo_zone_id,incident_date,incident_type,severity,fault_determination,claim_amount_usd,settlement_amount_usd,deductible_paid_usd,injuries_count,property_damage_flag,police_report_filed,legal_action_flag,investigation_status,resolution_date,repair_cost_usd,liability_pct,reimbursement_status,incident_code
500,118889,546577,739381,2023-04-27,Sensor Failure,Minor,Third Party At Fault,4096.56,1547.71,1500,1,0,1,0,Litigated,2022-05-10,5810.56,50,Reimbursed,INC-F
1976,118889,646469,152895,2022-03-24,Minor Scratch,Moderate,No Fault,1126.50,2103.09,500,0,1,1,0,Open,2022-12-04,4029.89,100,Reimbursed,INC-E
4236,448377,914504,567724,2023-11-07,Minor Scratch,Moderate,Driver At Fault,526.01,8843.57,2500,0,0,0,0,Closed,2024-03-06,334.35,75,Pending,INC-I
925,448377,885758,919204,2022-06-10,Theft,Minor,Driver At Fault,4542.76,4553.03,2000,0,1,1,0,Under Review,2024-08-05,273.06,0,Denied,INC-I


## 3. Investigasi Kendaraan: `ds1_vehicles` vs `ds3_fleet_vehicles`
Kedua dataset ini sama-sama berisi data kendaraan. Mari kita cek:
- Apakah ID kendaraan saling tumpang tindih?
- Apakah mereka memiliki VIN yang sama atau merupakan entitas berbeda?
- Apakah tipe kolomnya sama?

In [4]:
print(f"Tipe ID di ds1_vehicles: {vehicles_ds1['vehicle_id'].min()} s/d {vehicles_ds1['vehicle_id'].max()}")
print(f"Tipe ID di ds3_fleet_vehicles: {fleet_vehicles_ds3['fleet_vehicle_id'].min()} s/d {fleet_vehicles_ds3['fleet_vehicle_id'].max()}")

# Apakah vehicle_id di ds1 ada yang beririsan dengan fleet_vehicle_id di ds3?
overlap_ids = set(vehicles_ds1['vehicle_id']).intersection(set(fleet_vehicles_ds3['fleet_vehicle_id']))
print(f"\nJumlah ID Kendaraan yang Beririsan: {len(overlap_ids)}")

# Mari kita bandingkan data dari ID yang sama jika ada beririsan
if len(overlap_ids) > 0:
    sample_id = list(overlap_ids)[0]
    print(f"\nMembandingkan kendaraan ID {sample_id} antara ds1 dan ds3:")
    print("\n--- ds1_vehicles ---")
    display(vehicles_ds1[vehicles_ds1['vehicle_id'] == sample_id])
    print("\n--- ds3_fleet_vehicles ---")
    display(fleet_vehicles_ds3[fleet_vehicles_ds3['fleet_vehicle_id'] == sample_id])

Tipe ID di ds1_vehicles: 100000 s/d 926080
Tipe ID di ds3_fleet_vehicles: 100416 s/d 921984

Jumlah ID Kendaraan yang Beririsan: 40

Membandingkan kendaraan ID 892800 antara ds1 dan ds3:

--- ds1_vehicles ---


,vehicle_id,make,model,year,color,vehicle_type,home_city,battery_capacity_kwh,range_km,purchase_price_usd,monthly_insurance_usd,registration_state,fleet_zone,operational_status,last_maintenance_date,total_trips_completed,total_km_driven,charge_level_pct,assigned_depot,depreciation_rate_pct
4828,892800,Tesla,Model 3,2024,White,Compact,Chicago,75,521,94539,569.02,NY,Zone-B,Active,2023-06-16,7925,101810,37,Depot-Central,10.59



--- ds3_fleet_vehicles ---


,fleet_vehicle_id,vin,make,model,year,acquisition_date,acquisition_cost_usd,current_value_usd,odometer_km,battery_health_pct,software_version,hardware_revision,depot_location,operational_zone,vehicle_class,lease_or_owned,lease_expiry_date,compliance_status,last_inspection_date,decommission_flag
4807,892800,WBA87301586933,Waymo,EV6,2023,2019-12-04,86093,18408,237593,86,v5.0,HW4.1,Depot-West,Zone-E,Cargo,Financed,2024-03-07,Compliant,2024-09-02,0


## 4. Integritas Referensial (Foreign Keys Check)
Apakah semua transaksi, perjalanan, pemeliharaan, dan insiden merujuk ke entitas induk yang VALID?

In [5]:
# 4.1 Trips -> Vehicles (ds1)
missing_vehicles_in_trips = trips[~trips['vehicle_id'].isin(vehicles_ds1['vehicle_id'])]
print(f"Trip dengan vehicle_id yang TIDAK ditemukan di ds1_vehicles: {len(missing_vehicles_in_trips)}")

# 4.2 Trips -> Customers (ds2)
missing_customers_in_trips = trips[~trips['customer_id'].isin(customers['customer_id'])]
print(f"Trip dengan customer_id yang TIDAK ditemukan di ds2_customers: {len(missing_customers_in_trips)}")

# 4.3 Transactions -> Customers (ds2)
missing_customers_in_tx = transactions[~transactions['customer_id'].isin(customers['customer_id'])]
print(f"Transaksi dengan customer_id yang TIDAK ditemukan di ds2_customers: {len(missing_customers_in_tx)}")

# 4.4 Maintenance -> Fleet Vehicles (ds3)
missing_vehicles_in_maint = maintenance[~maintenance['fleet_vehicle_id'].isin(fleet_vehicles_ds3['fleet_vehicle_id'])]
print(f"Pemeliharaan dengan fleet_vehicle_id yang TIDAK ditemukan di ds3_fleet_vehicles: {len(missing_vehicles_in_maint)}")

# 4.5 Incidents -> Insurance Policies (ds4)
missing_policies_in_incidents = incidents[~incidents['policy_id'].isin(insurance['policy_id'])]
print(f"Insiden dengan policy_id yang TIDAK ditemukan di ds4_insurance_policies: {len(missing_policies_in_incidents)}")

Trip dengan vehicle_id yang TIDAK ditemukan di ds1_vehicles: 200
Trip dengan customer_id yang TIDAK ditemukan di ds2_customers: 4980
Transaksi dengan customer_id yang TIDAK ditemukan di ds2_customers: 200
Pemeliharaan dengan fleet_vehicle_id yang TIDAK ditemukan di ds3_fleet_vehicles: 200
Insiden dengan policy_id yang TIDAK ditemukan di ds4_insurance_policies: 200


## 5. Pemeriksaan Logika Bisnis & Anomali Nilai
Mari kita cari keganjilan dari nilai angka dan tanggal.

In [6]:
print("=== 5.1 Cek Nilai Negatif pada Kolom Finansial ===")
print(f"Trips - fare_amount_usd < 0: {len(trips[trips['fare_amount_usd'] < 0])}")
print(f"Transactions - gross_amount_usd < 0: {len(transactions[transactions['gross_amount_usd'] < 0])}")
print(f"Transactions - net_revenue_usd < 0: {len(transactions[transactions['net_revenue_usd'] < 0])}")
print(f"Maintenance - total_cost_usd < 0: {len(maintenance[maintenance['total_cost_usd'] < 0])}")
print(f"Incidents - claim_amount_usd < 0: {len(incidents[incidents['claim_amount_usd'] < 0])}")

print("\n=== 5.2 Cek Konsistensi Logika Tanggal ===")
# Trips: trip_end_time harus setelah trip_start_time
trips_date_err = pd.to_datetime(trips['trip_end_time']) < pd.to_datetime(trips['trip_start_time'])
print(f"Trips - Waktu Selesai < Waktu Mulai: {trips_date_err.sum()}")

# Insurance: coverage_end_date harus setelah coverage_start_date
ins_date_err = pd.to_datetime(insurance['coverage_end_date']) < pd.to_datetime(insurance['coverage_start_date'])
print(f"Insurance - Tanggal Berakhir < Tanggal Mulai: {ins_date_err.sum()}")

# Maintenance: actual_hours vs estimated_hours
print(f"Maintenance - Downtime Hours < 0: {len(maintenance[maintenance['downtime_hours'] < 0])}")

# Maintenance: next_service_mileage harus lebih tinggi daripada mileage_at_service
maint_mileage_err = maintenance['next_service_mileage'] < maintenance['mileage_at_service']
print(f"Maintenance - Next Service Mileage < Mileage Saat Service: {maint_mileage_err.sum()}")

=== 5.1 Cek Nilai Negatif pada Kolom Finansial ===
Trips - fare_amount_usd < 0: 0
Transactions - gross_amount_usd < 0: 0
Transactions - net_revenue_usd < 0: 0
Maintenance - total_cost_usd < 0: 0
Incidents - claim_amount_usd < 0: 0

=== 5.2 Cek Konsistensi Logika Tanggal ===
Trips - Waktu Selesai < Waktu Mulai: 2466
Insurance - Tanggal Berakhir < Tanggal Mulai: 0
Maintenance - Downtime Hours < 0: 0
Maintenance - Next Service Mileage < Mileage Saat Service: 2315


## 6. Menyusun Ringkasan Temuan DQA
Mari kita buat ringkasan data kotor yang berhasil kita temukan agar siap dibersihkan di tahap selanjutnya.

In [7]:
print("Proses Data Quality Assessment selesai! Silakan diskusikan hasil temuan di atas.")

Proses Data Quality Assessment selesai! Silakan diskusikan hasil temuan di atas.
